# Sample pass attribution

Browse makes with their traced passer and scorer, and animate a play to check it by eye.

`scripts/attribute_passers.py` traces who last held the ball before the shooter, from tracking proximity alone (`src/nba_assists/passer.py`), and writes `data/derived/passers.parquet`. For an assisted make, `passer_id` is just the known assister; `traced_passer_id` is always the proximity-based trace (even on assisted makes), so you can see where the two agree or not.

The animation (reusing `scripts/animate_window.py`) shows two new notes above the court, each matching the ring color on that player's circle:
- **scorer** (gold ring): the play's shooter.
- **passer** (blue ring): `passer_id` -- `(assisted)` when it's the known assister, `(traced)` when it came from the proximity trace, or "none identified" when no passer could be traced.

Run this with the project's `.venv` as the kernel, after `scripts/attribute_passers.py` has been run at least once.

In [ ]:
import sys
from pathlib import Path

import matplotlib
import pandas as pd
from IPython.display import HTML

# windows run up to windows.max_window_s (24s) at 25 Hz -- the default 20MB
# embed_limit silently truncates an animation's frames above that size.
matplotlib.rcParams["animation.embed_limit"] = 200

# scripts/ isn't an installed package -- make it (and scripts/vendor/, for
# Team.py) importable the same way the scripts themselves reach src/.
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

from animate_window import build_event_animation
from nba_assists.config import get_config

cfg = get_config()
pd.set_option("display.max_colwidth", 80)

## Pick a play

A mix of plays to browse: `agree` is only meaningful for assisted makes (whether `traced_passer_id` matches the known assister). Edit the filter below to focus on disagreements, NONEs, `trace_problem`, a specific game, etc.

In [ ]:
passers = pd.read_parquet(cfg.get_path("derived") / "passers.parquet")
makes = pd.read_parquet(cfg.get_path("derived") / "makes.parquet", columns=["game_id", "eventnum", "description"])

plays = passers.merge(makes, on=["game_id", "eventnum"])
plays["agree"] = plays["assisted"] & (plays["traced_passer_id"] == plays["assister_id"])

cols = [
    "game_id", "eventnum", "description", "assisted", "assister_id",
    "traced_passer_id", "passer_id", "agree", "trace_problem", "is_train_game",
]

# a mix of plays, so you can see agreements, disagreements, and NONEs while browsing
plays[cols].sample(10, random_state=0)

In [ ]:
# example: assisted makes where the trace disagreed with the known assister
disagreements = plays[plays["assisted"] & ~plays["agree"]]
disagreements[cols].head(20)

In [ ]:
num = 217  # row label from one of the tables above -- edit and re-run
GAME_ID = "0021500005" #plays.loc[num, "game_id"]
EVENTNUM = 250 #int(plays.loc[num, "eventnum"])
print(GAME_ID, EVENTNUM)

## Animate it

In [ ]:
anim = build_event_animation(GAME_ID, EVENTNUM, cfg=cfg)
HTML(anim.to_jshtml())